# 2.2 State, gates and skip: carrying context, letting through, going around

Notebook 2.1 gave you the kernel: a few weights that look at neighbours. This notebook
gives you the other three blocks of lesson 2, on the same heartbeats and read the same
way, against a floor in sigmas:

| section | the question | goal |
|---|---|---|
| 2.2.1 | what does a skip buy, and where? | 2.1 |
| 2.2.2 | how do you carry context along an axis, and what does it cost? | 2.5, 2.7 |
| 2.2.3 | what does a gate decide? | 2.6 |
| 2.2.4 | a gate on whole channels: squeeze-excite | 2.6 |

How a GRU or an LSTM looks inside is not the subject: your agent knows. What each block
assumes about the data, what it costs, and whether it paid here, is.

In [ ]:
import sys
import time

import heartbeats as hb
import matplotlib.pyplot as plt
import numpy as np
import torch
from loguru import logger
from notebooktester import param
from torch import nn
from vectormesh.components import Conv1D, Gate, Highway, Pool1D, Recurrent, Skip

logger.remove()
logger.add(sys.stderr, level="WARNING")

EPOCHS = param(8, test=1)
REPEATS = param(3, test=1)
MAX_ROWS = param(None, test=2000)  # fit rows; None keeps all of them

fit, val = hb.load(max_rows=MAX_ROWS)
print(f"fit  {len(fit):6d} beats  {fit.counts()}")
print(f"val  {len(val):6d} beats  {val.counts()}")


def block(channels_in: int, channels: int, kernel_size: int = 5) -> nn.Sequential:
    """A kernel and a GELU that keep the length: `(B, S, C_in) -> (B, S, C)`."""
    return nn.Sequential(
        Conv1D(channels_in, channels, kernel_size, padding=kernel_size // 2), nn.GELU()
    )


class Kernels(nn.Module):
    """Notebook 2.1's stack: `depth` blocks, each followed by a max pool of 2, flattened
    into a linear head. At depth 5 it is the design to beat."""

    def __init__(self, depth: int = 5, width: int = 32):
        super().__init__()
        layers, channels = [], 1
        for _ in range(depth):
            layers += [block(channels, width), Pool1D(2)]
            channels = width
        self.features = nn.Sequential(*layers)
        with torch.no_grad():
            flat = self.features(torch.zeros(1, hb.STEPS, 1)).flatten(1).shape[1]
        self.head = nn.Linear(flat, len(hb.CLASSES))

    def forward(self, x):
        return self.head(self.features(x).flatten(1))


floor, floor_s = hb.spread(Kernels, fit, val, EPOCHS, REPEATS)
print(hb.line("kernels, depth 5 (notebook 2.1)", floor, floor_s))

## 2.2.1 What a skip buys, and where

A skip is a route around a block whose output is added to the block's output:
`y = T(x) + x`. Because it is added, the shape stays (slide 10). It was invented for image
models (ResNet, 2015) and is now in nearly every architecture.

In lesson 1 the head had `skip` as an axis of the search. Open the class page,
`make dashboard LESSON=1`, and read that axis: what did a skip buy on DTD, and at which
depth? A head of one or two layers on a frozen encoder is a shallow stack.

The claim worth testing is the one on slide 10: **a skip promises the most in a deep
stack**, because the block only has to learn a correction to its input. Below, a stem
(a kernel and two pools, `(B, 187, 1) -> (B, 46, 32)`), then a stack of blocks at the
same width, then the linear head. Each block is wrapped in nothing, in a `Skip`, or in a
`Highway` (a skip with a gate on it, section 2.2.3), at 2 blocks and at 16.

**Before you run the next cell**, write down what you expect at 2 blocks and at 16, for
each of the three.

In [ ]:
class Deep(nn.Module):
    """A stem, then `n` blocks of width `width` on 46 steps, each wrapped in `wrap`."""

    def __init__(self, n: int, wrap: str = "none", width: int = 32):
        super().__init__()
        self.stem = nn.Sequential(block(1, width), Pool1D(2), Pool1D(2))
        blocks = []
        for _ in range(n):
            transform = block(width, width)
            if wrap == "skip":
                transform = Skip(transform, in_size=width)
            elif wrap == "highway":
                transform = Highway(transform, hidden_size=width)
            blocks.append(transform)
        self.body = nn.Sequential(*blocks)
        self.head = nn.Linear(width * 46, len(hb.CLASSES))

    def forward(self, x):
        return self.head(self.body(self.stem(x)).flatten(1))


deep = {}
for n in (2, 16):
    for wrap in ("none", "skip", "highway"):
        values, seconds = hb.spread(
            lambda n=n, wrap=wrap: Deep(n, wrap), fit, val, EPOCHS, REPEATS
        )
        deep[n, wrap] = values
        print(hb.line(f"{n:2d} blocks, {wrap}", values, seconds, versus=floor))

Compare with what you wrote down. Look at the plain stack at 16 blocks first: if its
macro recall is exactly 0.333, it says N for every beat, which is what a stack that
stopped learning does. The same 16 blocks with a skip around each still train: the
skip gives the gradient a route that does not pass through every block.

At 2 blocks, read the three in sigmas against each other. What a skip promises in a
deep stack it need not deliver in a shallow one; that is the same question as lesson 1's
skip axis on DTD.

`Skip` in vectormesh normalises its input first and then adds:
`transform(norm(x)) + norm(x)`. That norm is part of why the deep stack trains; ask your
agent what it does, if you want the details.

## 2.2.2 Carrying context along an axis: the state

A kernel looks at a few neighbours. Whatever lies further away, it only reaches through
depth. A **state** walks the axis step by step and carries what it has seen so far
(slides 8 and 9): at every step the same weights combine the new input with the state,
and hand the result to the next step.

`Recurrent(in_size, hidden_size, kind="gru")` takes `(B, S, D)` and gives `(B, S, H)`:
the state after every step. The state after the last step has seen the whole sequence.

In [ ]:
gru = Recurrent(in_size=32, hidden_size=64, kind="gru")
x = torch.randn(8, 46, 32)
states = gru(x)
print(
    f"in {tuple(x.shape)} -> states {tuple(states.shape)} -> last {tuple(states[:, -1].shape)}"
)

What it can, a kernel cannot: a relation over any distance, since the first step still
counts at the last; and order, since two inputs swapped lead the state elsewhere. What
it costs: step 10 waits for step 9, so the steps cannot be computed side by side, and
every step writes over a bit of the past.

The cost is easy to measure: one forward and backward pass over a batch, on the raw 187
steps and after the stem has pooled them to 46.

In [ ]:
def seconds_per_batch(model: nn.Module, x: torch.Tensor, n: int = 5) -> float:
    start = time.perf_counter()
    for _ in range(n):
        model(x).sum().backward()
    return (time.perf_counter() - start) / n


raw = torch.randn(128, hb.STEPS, 1)
pooled = torch.randn(128, 46, 32)
print(
    f"GRU over 187 steps: {seconds_per_batch(Recurrent(1, 64), raw) * 1000:6.1f} ms per batch"
)
print(
    f"GRU over  46 steps: {seconds_per_batch(Recurrent(32, 64), pooled) * 1000:6.1f} ms per batch"
)
print(
    f"kernel over 187 steps: {seconds_per_batch(block(1, 64), raw) * 1000:6.1f} ms per batch"
)

That is why a state usually comes after kernels that shortened the axis.

### Does the bet on order pay on a beat?

A state bets on **order**: what came before changes what this means (slide 16). Below,
kernels that shorten the beat, then a GRU whose last state goes to the head, against
notebook 2.1's kernel stack of depth 5. Both see the whole beat: the stack through its
receptive field of 156 steps, the GRU through its state.

**Before you run the next cell**, write down what you expect, and which of the two you
expect to take longer.

In [ ]:
class KernelsThenState(nn.Module):
    """`depth` kernel blocks with pools, then a GRU; its last state goes to the head."""

    def __init__(self, depth: int = 2, width: int = 32, hidden: int = 64):
        super().__init__()
        layers, channels = [], 1
        for _ in range(depth):
            layers += [block(channels, width), Pool1D(2)]
            channels = width
        self.features = nn.Sequential(*layers)
        self.state = Recurrent(width, hidden, kind="gru")
        self.head = nn.Linear(hidden, len(hb.CLASSES))

    def forward(self, x):
        return self.head(self.state(self.features(x))[:, -1])


state, state_s = hb.spread(KernelsThenState, fit, val, EPOCHS, REPEATS)
print(hb.line("kernels, depth 5", floor, floor_s))
print(hb.line("kernels, depth 2, then a GRU", state, state_s, versus=floor))

Compare with what you wrote down, the seconds included. Where does the bet on order
pay? On slide 16: *a rhythm across beats, a sentence*. Every row of this dataset is one
beat, cut out of its recording. Inside one beat the order that matters is short, and a
deep enough stack of kernels sees it all. The order that tells an S beat from an N, the
beat coming too early after the one before, lies between beats, and one row does not
hold it.

That is a property of the data, not of the GRU. A block is a bet about the data: before
you pay for a state, ask what order there is to carry.

## 2.2.3 What a gate decides

A gate is a learned multiplier between 0 and 1 for every element, computed by the
network from its own input with a sigmoid: `y = sigmoid(W x) * x` (slide 11). Where it is
open the input passes; where it is closed the output is zero. The shape stays.

The same multiplier, put between two routes, is what the other blocks of this lesson
are made of (slide 15):

| block | formula | the gate chooses between |
|---|---|---|
| GRU | `Γ * h_new + (1 - Γ) * h_old` | a new state and the old one |
| Highway | `Γ * T(x) + (1 - Γ) * x` | a block and its own input |
| Skip | `T(x) + x` | nothing: there is no gate |

Below, notebook 2.1's stack of depth 3 with a `Gate` on its output, trained once. Then
its gate is read on one N beat and one V beat: per position, the mean of the gate over
the 32 channels.

In [ ]:
class GatedKernels(nn.Module):
    """Three kernel blocks with pools, a gate on their output, a linear head."""

    def __init__(self, width: int = 32):
        super().__init__()
        layers, channels = [], 1
        for _ in range(3):
            layers += [block(channels, width), Pool1D(2)]
            channels = width
        self.features = nn.Sequential(*layers)
        self.gate = Gate(width)
        self.head = nn.Linear(width * 23, len(hb.CLASSES))

    def forward(self, x):
        return self.head(self.gate(self.features(x)).flatten(1))

    def openness(self, x):
        """The gate's multipliers for `x`: `(B, S', C)`, every value between 0 and 1."""
        return torch.sigmoid(self.gate.project(self.features(x)))


gated = GatedKernels()
result, _ = hb.train(gated, fit, val, EPOCHS)
print(
    "one run of the gated stack:", {k: round(v, 3) for k, v in result["recall"].items()}
)

gated = gated.cpu().eval()
labels = fit.labels.numpy()
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
for ax, k in zip(axes, (0, 2), strict=True):
    b = fit.beats[np.flatnonzero(labels == k)[0]]
    with torch.no_grad():
        g = gated.openness(b.unsqueeze(0))[0].mean(dim=1)  # (23,)
    ax.plot(
        np.arange(hb.STEPS) / 125, b[:, 0], color="0.6", label=f"a {hb.CLASSES[k]} beat"
    )
    centres = (np.arange(len(g)) + 0.5) * hb.STEPS / len(g) / 125
    ax.step(centres, g, where="mid", color="C1", lw=2, label="gate, mean over channels")
    ax.set_ylim(0, 1.05)
    ax.legend(loc="upper right")
axes[1].set_xlabel("seconds after the R-peak")
plt.tight_layout()
plt.show()

The gate is computed from the input, so it differs per beat: it is the network deciding,
per position and per channel, what passes. Read the two pictures: where is it open, and
is that where the two beats differ? A gate that is open everywhere has learned nothing
to filter; it costs its weights and passes the input on.

## 2.2.4 A gate on whole channels: squeeze-excite

Squeeze-excite is a gate on the channels of a kernel's output (slide 14). **Squeeze**:
pool away the sequence axis, so every channel is one number, how strongly it answered
anywhere. **Excite**: a narrow network, `C -> C/r -> C`, then a sigmoid. The result
multiplies whole channels. The bottleneck is narrow on purpose: from a summary, per
channel, it has to guess how useful that channel is, given the others.

vectormesh has no squeeze-excite of its own. It is a few lines, and writing it is a
good test of whether you can read the shapes:

In [ ]:
class SqueezeExcite(nn.Module):
    """A gate on whole channels: `(B, S, C) -> (B, S, C)`."""

    def __init__(self, channels: int, reduction: int = 4):
        super().__init__()
        self.excite = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.GELU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        squeezed = x.mean(dim=1)  # (B, S, C) -> (B, C): where is gone
        gate = self.excite(squeezed)  # (B, C), every value between 0 and 1
        return x * gate.unsqueeze(1)  # (B, 1, C) broadcasts over S


se = SqueezeExcite(32)
x = torch.randn(8, 46, 32)
print(f"in {tuple(x.shape)} -> out {tuple(se(x).shape)}, {hb.n_params(se)} weights")


class ExcitedKernels(Kernels):
    """Notebook 2.1's stack of depth 5, with squeeze-excite after every block."""

    def __init__(self, depth: int = 5, width: int = 32):
        super().__init__(depth, width)
        layers, channels = [], 1
        for _ in range(depth):
            layers += [block(channels, width), SqueezeExcite(width), Pool1D(2)]
            channels = width
        self.features = nn.Sequential(*layers)

**Before you run the next cell**, write down what you expect squeeze-excite to add to
the stack of depth 5.

In [ ]:
excited, excited_s = hb.spread(ExcitedKernels, fit, val, EPOCHS, REPEATS)
print(hb.line("kernels, depth 5", floor, floor_s))
print(hb.line("kernels, depth 5, squeeze-excite", excited, excited_s, versus=floor))

Inside about two sigmas, the difference is no result, whatever its sign; outside, it is
one, also when it is negative. Lesson 3 takes the line from gate to highway to mixture of
experts further (notebook 3.1), and asks the same question of every one of them: what
did the extra weights buy?

## What you now know

- A skip adds a block's input to its output; the shape stays. In a deep stack it is the
  difference between a model that trains and one that says N for every beat.
- A state carries context along an ordered axis: any distance and order, paid in steps
  that wait for each other. It pays where there is order to carry; inside one cut-out
  beat there is little that a deep stack of kernels does not see.
- A gate is a learned multiplier between 0 and 1, computed from the input; the GRU and
  the highway put it between two routes, the skip leaves it out, and squeeze-excite puts
  it on whole channels.
- Every block is a bet about the data. Whether it paid is a measurement, in sigmas, on
  patients the model has not seen.